# TP6 — IA Défensive : Détecter les attaques réseau par Machine Learning
**Cours IA & Cybersécurité — Master 1**

---

## Ce que vous allez apprendre

Dans les TPs précédents, vous avez :
- Construit des règles SI/ALORS manuellement (TP_S1)
- Détecté des anomalies avec K-Means et Isolation Forest (TP2)
- Entraîné un réseau de neurones sur des données simulées (TP4)

Dans ce TP, vous allez appliquer le **Machine Learning sur de vrais logs réseau** téléchargés depuis Kaggle pour construire un système de détection d'intrusion (*Intrusion Detection System*, **IDS**) basé sur l'IA.

À la fin de ce TP, vous saurez :
1. Télécharger et préparer un dataset réseau réel depuis Kaggle
2. Choisir et entraîner un classifieur adapté (**Random Forest**)
3. Évaluer rigoureusement un IDS (précision, rappel, F1, matrice de confusion)
4. Comprendre les limites de l'IA défensive face aux attaques adversariales

**Durée estimée** : 2h30  
**Prérequis** : TP2 (UEBA) et TP4 (MLP) complétés

---

## Partie 0 — Téléchargement du dataset Kaggle

Nous utilisons le **NSL-KDD dataset**, une référence académique pour les IDS.  
Il contient des connexions réseau annotées issues du projet KDD Cup 1999, nettoyées et équilibrées.

> **Pourquoi NSL-KDD ?** C'est le benchmark standard pour évaluer les systèmes IDS. Vous verrez ce dataset cité dans les articles de recherche en cybersécurité.

> 💡 **Prérequis** : votre fichier `~/.config/kaggle/kaggle.json` doit être en place (voir TP5 section 0.1).

In [ ]:
import os, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Vérification de la clé Kaggle
kaggle_path = os.path.expanduser("~/.config/kaggle/kaggle.json")
if not os.path.exists(kaggle_path):
    print("❌ Clé Kaggle introuvable — consultez TP5 section 0.1")
else:
    print("✅ Clé Kaggle trouvée")

# Téléchargement du dataset NSL-KDD
os.makedirs("data", exist_ok=True)
print("\nTéléchargement du dataset NSL-KDD...")

result = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "hassan06/nslkdd",
     "--unzip", "-p", "data/"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ Dataset téléchargé !")
    print(f"   Fichiers : {os.listdir('data/')}")
else:
    print("❌ Erreur :", result.stderr[:200])

---
## Partie 1 — Comprendre les données

### 1.1 Chargement et structure

Le dataset NSL-KDD décrit chaque connexion réseau avec **41 caractéristiques** :
- **Caractéristiques de base** : durée, protocole, service, octets envoyés/reçus...
- **Caractéristiques de contenu** : nombre de tentatives de login, accès root...
- **Caractéristiques temporelles** : connexions vers le même hôte dans les 2 dernières secondes...

La colonne cible `label` peut valoir :
- `normal` : connexion légitime
- `neptune`, `smurf`, `portsweep`... : types d'attaques

In [ ]:
# Noms des colonnes NSL-KDD (le fichier CSV n'a pas d'en-tête)
COLONNES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

# Chargement — adapter le nom de fichier si nécessaire
try:
    df = pd.read_csv("data/KDDTrain+.txt", names=COLONNES)
    print(f"✅ Dataset chargé : {df.shape[0]:,} connexions × {df.shape[1]} colonnes")
except FileNotFoundError:
    # Cherche le bon nom de fichier
    fichiers = [f for f in os.listdir('data/') if f.endswith('.txt') or f.endswith('.csv')]
    print(f"Fichiers trouvés : {fichiers}")
    print("Modifiez le nom de fichier ci-dessus en conséquence.")

df.head(3)

In [ ]:
# Distribution des labels (types de trafic)
print("Types de trafic dans le dataset :")
print(df['label'].value_counts())

# Visualisation
top_labels = df['label'].value_counts().head(10)
plt.figure(figsize=(10, 4))
top_labels.plot(kind='bar', color=sns.color_palette('Set2', len(top_labels)))
plt.title("Top 10 types de trafic — NSL-KDD")
plt.xlabel("Type")
plt.ylabel("Nombre de connexions")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 1.2 Simplification : problème binaire

NSL-KDD contient des dizaines de types d'attaques. Pour commencer, nous allons simplifier en **classification binaire** : `normal` vs `attaque`.

C'est la base de tout IDS : détecter si une connexion est malveillante, quelle que soit l'attaque précise.

In [ ]:
# ============================================================
#  À COMPLÉTER
# ============================================================
# Créez une colonne 'cible' binaire :
#   - 0 si label == 'normal'
#   - 1 sinon (toute attaque)
# Indice : utilisez np.where(condition, valeur_si_vrai, valeur_si_faux)

df['cible'] = np.where(df['label'] == ???, 0, ???)

nb_normal  = (df['cible'] == 0).sum()
nb_attaque = (df['cible'] == 1).sum()
print(f"Connexions normales  : {nb_normal:,} ({nb_normal/len(df)*100:.1f}%)")
print(f"Connexions attaquées : {nb_attaque:,} ({nb_attaque/len(df)*100:.1f}%)")

---
## Partie 2 — Préparation des données (Feature Engineering)

### 2.1 Encoder les variables catégorielles

Les algorithmes de ML travaillent avec des **nombres**. Or certaines colonnes sont du texte :
- `protocol_type` : `tcp`, `udp`, `icmp`
- `service` : `http`, `ftp`, `ssh`...
- `flag` : `SF`, `REJ`, `S0`...

Il faut les convertir en nombres — c'est le **label encoding**.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Copie de travail
df_ml = df.copy()

# Colonnes catégorielles à encoder
cols_cat = ['protocol_type', 'service', 'flag']

# ============================================================
#  À COMPLÉTER
# ============================================================
# Appliquez LabelEncoder sur chaque colonne catégorielle.
# Indice : LabelEncoder().fit_transform(série)

encoders = {}
for col in cols_cat:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[???])  # quelle colonne encoder ?
    encoders[col] = le
    print(f"{col} : {list(le.classes_)} → {list(range(len(le.classes_)))}")

print("\n✅ Encodage terminé")

### 2.2 Sélection des caractéristiques

41 caractéristiques, c'est beaucoup. Nous allons garder les plus pertinentes pour la détection d'intrusion — celles qui varient significativement entre trafic normal et attaques.

In [ ]:
# Caractéristiques sélectionnées (bonnes pour la détection d'intrusion)
FEATURES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'logged_in', 'num_failed_logins',
    'count', 'srv_count', 'serror_rate', 'rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_serror_rate', 'dst_host_rerror_rate'
]

X = df_ml[FEATURES].values.astype(np.float32)
y = df_ml['cible'].values

print(f"Matrice de features X : {X.shape}")
print(f"Vecteur cible y       : {y.shape} — valeurs : {np.unique(y)}")

# Aperçu statistique
pd.DataFrame(X, columns=FEATURES).describe().round(2)

### 2.3 Découpage train / test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ============================================================
#  À COMPLÉTER
# ============================================================
# Découpez X et y en train (80%) et test (20%).
# Utilisez random_state=42 pour la reproductibilité.

X_train, X_test, y_train, y_test = train_test_split(
    ???, ???, test_size=???, random_state=42
)

# Normalisation
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train : {X_train.shape[0]:,} connexions")
print(f"Test  : {X_test.shape[0]:,} connexions")
print(f"Ratio attaques dans le train : {y_train.mean()*100:.1f}%")

---
## Partie 3 — Entraîner le classifieur : Random Forest

### 3.1 Pourquoi Random Forest ?

Le **Random Forest** (*Forêt aléatoire*) est un algorithme très utilisé en cybersécurité pour sa robustesse et son interprétabilité.

> **Analogie** : imaginez 100 experts en sécurité analysant indépendamment la même connexion. Chacun donne son verdict (normal / attaque). La décision finale est le **vote majoritaire**. C'est exactement ce que fait un Random Forest — 100 arbres de décision, chacun entraîné sur un sous-ensemble aléatoire des données.

**Avantages :**
- Résistant au surapprentissage (*overfitting*)
- Fonctionne bien sans normalisation (mais on la garde pour la comparaison)
- Indique l'importance de chaque caractéristique
- Rapide à entraîner

### 3.2 Entraînement

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

# ============================================================
#  À COMPLÉTER
# ============================================================
# Créez et entraînez un RandomForestClassifier.
# Paramètres : n_estimators=100 (100 arbres), random_state=42
# Méthode d'entraînement : .fit(X_train, y_train)

print("Entraînement du Random Forest...")
debut = time.time()

rf = RandomForestClassifier(n_estimators=???, random_state=???)
rf.???(X_train, y_train)

duree = time.time() - debut
print(f"✅ Entraînement terminé en {duree:.1f}s")
print(f"   Nombre d'arbres : {rf.n_estimators}")
print(f"   Profondeur max observée : {max(e.get_depth() for e in rf.estimators_)}")

---
## Partie 4 — Évaluation du système IDS

### 4.1 Les métriques d'un IDS

Pour un IDS, la **précision** seule ne suffit pas. Un modèle qui prédit toujours "attaque" aurait 50% de précision — mais serait inutilisable.

Les métriques clés sont :

| Métrique | Formule | Signification |
|----------|---------|---------------|
| **Précision** (*Precision*) | TP / (TP + FP) | Parmi les alertes, combien sont réelles ? |
| **Rappel** (*Recall*) | TP / (TP + FN) | Parmi les attaques réelles, combien sont détectées ? |
| **F1-score** | 2 × P × R / (P + R) | Compromis précision/rappel |
| **Taux de faux positifs** | FP / (FP + TN) | Fausses alarmes |

> **En cybersécurité**, on préfère souvent maximiser le **rappel** (ne rater aucune attaque) au détriment de la précision (accepter plus de fausses alarmes).

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

# Prédictions
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]  # probabilité d'être une attaque

# ============================================================
#  À COMPLÉTER
# ============================================================
# Affichez le rapport de classification.
# Utilisez classification_report(y_test, y_pred, target_names=[...])

print("=== Rapport de classification ===")
print(classification_report(???, ???, target_names=['Normal', 'Attaque']))

# AUC-ROC
auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC : {auc:.4f}  (1.0 = parfait, 0.5 = aléatoire)")

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Matrice de confusion
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['Normal', 'Attaque'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Normal', 'Attaque'])
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
ax.set_title('Matrice de confusion')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=13)

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1], 'k--', linewidth=1, label='Aléatoire')
axes[1].set_xlabel('Taux de faux positifs')
axes[1].set_ylabel('Taux de vrais positifs (Rappel)')
axes[1].set_title('Courbe ROC')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Résumé :")
print(f"   Vrais Positifs  (TP) : {tp:,}  — attaques correctement détectées")
print(f"   Faux Positifs   (FP) : {fp:,}  — fausses alarmes")
print(f"   Faux Négatifs   (FN) : {fn:,}  — attaques manquées ← critique")
print(f"   Vrais Négatifs  (TN) : {tn:,}  — trafic normal correctement classé")
print(f"\n   Taux de détection  : {tp/(tp+fn)*100:.2f}%")
print(f"   Taux de faux positifs : {fp/(fp+tn)*100:.2f}%")

### 4.2 Importance des caractéristiques

L'un des avantages du Random Forest est d'indiquer quelles caractéristiques sont les plus utiles pour la décision. C'est essentiel pour l'**explicabilité** d'un IDS — un analyste SOC doit comprendre pourquoi une alerte est levée.

In [ ]:
# Importance des caractéristiques
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 4))
plt.bar(range(len(FEATURES)), importances[indices],
        color=sns.color_palette('viridis', len(FEATURES)))
plt.xticks(range(len(FEATURES)),
           [FEATURES[i] for i in indices],
           rotation=45, ha='right', fontsize=8)
plt.title("Importance des caractéristiques — Random Forest IDS")
plt.ylabel("Importance relative")
plt.tight_layout()
plt.show()

print("Top 5 caractéristiques les plus discriminantes :")
for i, idx in enumerate(indices[:5]):
    print(f"  {i+1}. {FEATURES[idx]:30s} : {importances[idx]:.4f}")

---
## Partie 5 — Tester sur de nouvelles connexions

### 5.1 Analyser des connexions individuelles

In [ ]:
def analyser_connexion_ids(features_dict: dict) -> None:
    """
    Analyse une connexion réseau avec notre IDS.
    features_dict : dictionnaire {nom_feature: valeur}
    """
    # Construction du vecteur de features dans le bon ordre
    vecteur = np.array([[features_dict.get(f, 0) for f in FEATURES]], dtype=np.float32)
    vecteur_normalise = scaler.transform(vecteur)

    # Prédiction
    prediction  = rf.predict(vecteur_normalise)[0]
    probabilite = rf.predict_proba(vecteur_normalise)[0][1]

    etiquette = "⚠️  ATTAQUE" if prediction == 1 else "✅ Normale"
    print(f"Décision       : {etiquette}")
    print(f"Probabilité    : {probabilite:.1%} de probabilité d'attaque")
    print()

# Test 1 : connexion HTTP normale typique
print("--- Connexion HTTP normale ---")
analyser_connexion_ids({
    'duration': 0, 'protocol_type': 2,  # tcp=2 après encodage
    'src_bytes': 232, 'dst_bytes': 8153,
    'logged_in': 1, 'count': 5, 'srv_count': 5,
    'serror_rate': 0.0, 'same_srv_rate': 1.0
})

# Test 2 : scan de ports (neptune-like)
print("--- Scan de ports suspect ---")
analyser_connexion_ids({
    'duration': 0, 'protocol_type': 2,
    'src_bytes': 0, 'dst_bytes': 0,
    'logged_in': 0, 'count': 511, 'srv_count': 511,
    'serror_rate': 1.0, 'same_srv_rate': 1.0,
    'dst_host_count': 255, 'dst_host_serror_rate': 1.0
})

# ============================================================
#  À COMPLÉTER
# ============================================================
# Inventez une 3ème connexion ambiguë et analysez-la.
# Que remarquez-vous ?
print("--- Connexion ambiguë (à vous) ---")
analyser_connexion_ids({
    'duration': ???,
    'src_bytes': ???,
    'dst_bytes': ???,
    # complétez avec d'autres features si vous le souhaitez
})

---
## Partie 6 — Les limites de l'IA défensive

### 6.1 Dérive de distribution (concept drift)

Un IDS entraîné sur des données de 1999 (NSL-KDD) ne détectera pas forcément les attaques de 2025. Les attaquants **évoluent** et les patterns changent.

> **Analogie** : un détecteur de visages entraîné sur des photos des années 90 aurait du mal avec les selfies d'aujourd'hui — la distribution des images a changé (angle, lumière, résolution).

In [ ]:
# Simulation d'une attaque "nouvelle génération" inconnue du modèle
# L'attaquant connaît le modèle et génère un trafic qui ressemble à du trafic normal

print("=== Simulation : attaque évasive ===")
print("L'attaquant a modifié son trafic pour éviter la détection...\n")

# Attaque classique (connue du modèle)
print("--- Attaque classique ---")
analyser_connexion_ids({
    'duration': 0, 'src_bytes': 0, 'dst_bytes': 0,
    'count': 500, 'srv_count': 500,
    'serror_rate': 1.0, 'dst_host_serror_rate': 1.0
})

# Même attaque, mais ralentie et fragmentée pour imiter du trafic normal
print("--- Même attaque, rendue furtive (slow scan) ---")
analyser_connexion_ids({
    'duration': 10,           # connexion plus longue = moins suspect
    'src_bytes': 150,         # quelques octets envoyés = moins suspect
    'dst_bytes': 400,
    'count': 3,               # faible count = ressemble à du trafic normal
    'srv_count': 2,
    'serror_rate': 0.3,       # taux d'erreur réduit
    'same_srv_rate': 0.67,
    'logged_in': 0
})

print("💡 Un attaquant qui connaît les features du modèle peut adapter son attaque.")
print("   C'est l'objet du TP sur les attaques adversariales (module offensif).")

### 6.2 Résumé des limites et contre-mesures

| Limite | Description | Contre-mesure |
|--------|-------------|---------------|
| **Concept drift** | Les attaques évoluent, le modèle vieillit | Réentraînement régulier, apprentissage en ligne |
| **Attaques évasives** | L'attaquant adapte son trafic | Détection d'anomalies + signatures (hybride) |
| **Faux positifs** | Alertes sur du trafic légitime | Ajustement du seuil de décision |
| **Données rares** | Nouveaux types d'attaques non vus à l'entraînement | Apprentissage few-shot, one-class classification |
| **Explicabilité** | L'analyste SOC ne comprend pas pourquoi | SHAP, LIME, importance des features |

In [ ]:
# Ajustement du seuil de décision
# Par défaut, le seuil est 0.5. On peut le modifier selon les priorités.

print("=== Impact du seuil de décision ===")
print()

for seuil in [0.3, 0.5, 0.7]:
    y_pred_seuil = (y_proba >= seuil).astype(int)
    cm_s = confusion_matrix(y_test, y_pred_seuil)
    tn_s, fp_s, fn_s, tp_s = cm_s.ravel()
    detection  = tp_s / (tp_s + fn_s) * 100
    faux_pos   = fp_s / (fp_s + tn_s) * 100
    print(f"Seuil {seuil} → Détection : {detection:.1f}%  |  Faux positifs : {faux_pos:.1f}%")

print()
print("Avec un seuil bas (0.3) : on détecte plus d'attaques, mais plus de fausses alarmes.")
print("Avec un seuil haut (0.7) : moins de fausses alarmes, mais des attaques passent.")
print("→ Le choix dépend du contexte opérationnel (SOC, système embarqué, criticité...).")

---
## Récapitulatif

| Concept | Ce que vous avez fait |
|---------|----------------------|
| **Kaggle / NSL-KDD** | Téléchargé et exploré un benchmark IDS académique réel |
| **Feature engineering** | Encodé les variables catégorielles, sélectionné les features |
| **Random Forest** | Entraîné un IDS robuste par vote d'arbres de décision |
| **Métriques IDS** | Analysé précision, rappel, F1, courbe ROC |
| **Importance features** | Identifié quelles variables révèlent le plus une attaque |
| **Attaques évasives** | Compris comment un attaquant peut tromper un IDS ML |
| **Seuil de décision** | Ajusté le compromis détection / fausses alarmes |

### Questions de réflexion

1. Votre Random Forest obtient un excellent score sur NSL-KDD. Serait-il aussi performant sur du trafic réseau de votre campus ? Pourquoi ?
2. La caractéristique la plus importante dans votre modèle, est-elle difficile à falsifier pour un attaquant ?
3. Dans un système de sécurité aéronautique (DO-326A), quelle contrainte supplémentaire s'imposerait à votre IDS ?
4. Quel est l'avantage de combiner un IDS à base de règles (TP_S1) et un IDS ML (ce TP) ?
5. Comment détecteriez-vous une attaque **lente et furtive** qui génère peu de connexions par heure ?

---
*TP6 — Cours IA & Cybersécurité · M1 · 2025-2026*